## Modélisation Perceptron, optimisation et sauvegarde

Je cherche ici le meilleur perceptron possible pour mon jeu de données, en optimisant ses principaux paramètres via GridSearchCV.  
Je reporte les performances sur le jeu de test pour garantir la bonne généralisation du modèle, puis je sauvegarde le pipeline final pour éventuel déploiement ou réutilisation.


In [3]:
# 1. Import et préparation des données (à mettre en début de notebook si pas encore fait)
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Perceptron
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X = data.data
y = data.target

# Je découpe le dataset pour avoir un train et un test bien séparés
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. Pipeline (standardisation + perceptron) + gridsearch pour optimisation
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", Perceptron())
])
params = {
    "clf__penalty": [None, "l2"],
    "clf__alpha": [0.0001, 0.001, 0.01],
    "clf__max_iter": [500, 1000],
    "clf__tol": [1e-3, 1e-4],
    "clf__shuffle": [True],
}
from sklearn.model_selection import GridSearchCV
search = GridSearchCV(pipe, params, cv=5, scoring="accuracy", n_jobs=-1)
search.fit(X_train, y_train)

# 3. Évaluation du modèle sur le jeu de test
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
y_pred = search.best_estimator_.predict(X_test)
print("Accuracy sur les données test :", accuracy_score(y_test, y_pred))
print("Matrice de confusion\n", confusion_matrix(y_test, y_pred))
print("Rapport de classification\n", classification_report(y_test, y_pred, target_names=data.target_names))

# 4. Sauvegarde du modèle optimisé
import joblib
joblib.dump(search.best_estimator_, "perceptron_best_model.pkl")
print("Modèle sauvegardé dans perceptron_best_model.pkl")


Accuracy sur les données test : 0.9883040935672515
Matrice de confusion
 [[ 62   1]
 [  1 107]]
Rapport de classification
               precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        63
      benign       0.99      0.99      0.99       108

    accuracy                           0.99       171
   macro avg       0.99      0.99      0.99       171
weighted avg       0.99      0.99      0.99       171

Modèle sauvegardé dans perceptron_best_model.pkl


### Interprétation et conclusion

La précision obtenue me donne une bonne confiance dans le modèle sur ces données.  
En sauvegardant le pipeline complet (scaler + perceptron optimisé), je peux facilement le réutiliser sur de nouveaux jeux ou pour une comparaison avec des modèles alternatifs.
